# Patient Identity Resolution — Enterprise (Splink)

Enterprise-grade probabilistic patient identity resolution using
[Splink](https://moj-analytical-services.github.io/splink/).

## Pipeline

1. **Prepare** — Load `silver.patients`, normalise, reject duplicate keys.
2. **Score** — Splink probabilistic scoring with blocking rules.
3. **Cluster** — Connected components on auto-merge + approved edges.
4. **Resolve** — Assign stable enterprise patient IDs, manage aliases.
5. **Outputs** — Patient map, golden dimension, tombstoning.
6. **Audit** — Validate invariants, write run audit.

All business logic lives in `src/identity/` and `config/identity_config.py`.
This notebook is a thin orchestrator.

In [ ]:
# No external library installs required.
# The probabilistic scoring in src/identity/matching.py uses only
# native PySpark functions (levenshtein, etc.) – no Splink dependency.
print("Environment ready.")

In [ ]:
from datetime import datetime, timezone

from config.identity_config import (
    IDENTITY_TABLES,
    AUTO_MERGE_THRESHOLD,
    REVIEW_MIN_THRESHOLD,
)
from src.identity.preparation import prepare_patients
from src.identity.matching    import score_pairs, build_clusters
from src.identity.resolution  import (
    resolve_enterprise_ids,
    build_analytical_outputs,
    validate_and_audit,
)

## 1. Parameters

In [ ]:
# Parameters (set FULL_REFRESH = True to reprocess all records)
try:
    FULL_REFRESH = dbutils.widgets.get("full_refresh").lower() == "true"
except Exception:
    FULL_REFRESH = False

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
tables = IDENTITY_TABLES

print(f"run_id       : {RUN_ID}")
print(f"full_refresh : {FULL_REFRESH}")
print(f"source       : {tables['source']}")

## 2. Prepare Patients

In [ ]:
valid_patients = prepare_patients(spark, tables, RUN_ID)

## 3. Probabilistic Scoring (Splink)

In [ ]:
scored_pairs = score_pairs(spark, valid_patients)

## 4. Build Clusters

In [ ]:
clusters = build_clusters(
    spark, scored_pairs, valid_patients, tables,
    AUTO_MERGE_THRESHOLD, REVIEW_MIN_THRESHOLD, RUN_ID,
)

## 5. Enterprise ID Resolution

In [ ]:
resolve_enterprise_ids(
    spark, clusters, scored_pairs, valid_patients, tables, RUN_ID,
)

## 6. Analytical Outputs

In [ ]:
build_analytical_outputs(spark, valid_patients, tables)

## 7. Validation & Run Audit

In [ ]:
summary = validate_and_audit(
    spark, valid_patients, tables, RUN_ID, FULL_REFRESH,
)
display(summary)

## 8. Operational Views

In [ ]:
from pyspark.sql import functions as F
from src.identity.writer import table_exists

display(
    spark.table(tables['patient_map'])
    .groupBy('match_status')
    .agg(
        F.count('*').alias('source_record_count'),
        F.countDistinct('enterprise_patient_id').alias('enterprise_patient_count'),
    )
)

display(
    spark.table(tables['registry'])
    .groupBy('enterprise_patient_id')
    .agg(
        F.count('*').alias('source_record_count'),
        F.countDistinct('source_branch').alias('source_branch_count'),
    )
    .orderBy(F.desc('source_record_count'))
    .limit(50)
)

if table_exists(spark, tables['review_queue']):
    display(
        spark.table(tables['review_queue'])
        .groupBy('review_status', 'review_reason')
        .agg(F.count('*').alias('pair_count'))
        .orderBy(F.desc('pair_count'))
    )